# Lesson 3: Agentic Search

In [1]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [3]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


"Nvidia's Blackwell GPU features the second-generation Transformer Engine and custom Tensor Core technology for AI workloads. It also includes a Decompression Engine and high-speed memory access."

## Regular search

In [4]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [5]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

returning previous results due to exception reaching ddg.
https://weather.com/weather/today/l/USCA0987:1:US
https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8


In [6]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [7]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

returning previous results due to exception reaching ddg.
Website: https://weather.com/weather/today/l/USCA0987:1:US


<body class="inter_3e69cff8-module__PPczxG__className font-sans"><script>(self.__next_s=self.__next_s||[]).push([0,{"children":"(function(){\n        try{\n          var cookies=document.cookie.split('; ');\n          var sessionCookie='';\n          for(var i=0;i<cookies.length;i++){\n            if(cookies[i].indexOf('wxu-metrics-session=')===0){\n              sessionCookie=cookies[i].slice(20);break;\n            }\n          }\n          var now=Date.now();\n          var la=parseInt(localStorage.getItem('metric-session-last-active-time')||'0',10);\n          var expired=la>0&&Math.abs(now-la)>1800000;\n          if(!sessionCookie||expired){\n            var id=crypto&&crypto.randomUUID ? crypto.randomUUID() : (function(){\n              return 'xxxxxxxx-xxxx-4xxx-yxxx-xxxxxxxxxxxx'.replace(/[xy]/g,function(c){\n                var r=Math.random()*16|0;return(c===

In [8]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/today/l/USCA0987:1:US


Home Forecast Radar Video Explore More Sign in San Francisco Weather Downtown, San Francisco, California Today's Outlook Things to do around San Francisco Daily Forecast Trending Now Tropical Storm Amanda forms in the Eastern Pacific UN: El Niño is 'arriving on our doorstep' How your weather forecast can help protect you from ticks 0:37 Entire North Carolina home moved to stop from crumbling into the sea Tropical Storm Amanda forms in the Eastern Pacific UN: El Niño is 'arriving on our doorstep' How your weather forecast can help protect you from ticks 0:37 Entire North Carolina home moved to stop from crumbling into the sea Home & Garden Homeowner's Guide To Termite Season Entire North Carolina home moved away from oceanfront 0:34 Entire Outer Banks home moved 0:20 Weathering The Damage: How To Prepare for Termite Season 0:58 Homeowner's Guide To Termite Season Entire North Carolina home moved away from oceanfront 0:34 Entir

## Agentic Search

In [9]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1780555668, 'localtime': '2026-06-03 23:47'}, 'current': {'last_updated_epoch': 1780554600, 'last_updated': '2026-06-03 23:30', 'temp_c': 13.9, 'temp_f': 57.0, 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': 1000}, 'wind_mph': 7.8, 'wind_kph': 12.6, 'wind_degree': 266, 'wind_dir': 'W', 'pressure_mb': 1016.0, 'pressure_in': 30.01, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 89, 'cloud': 25, 'feelslike_c': 13.0, 'feelslike_f': 55.3, 'windchill_c': 12.2, 'windchill_f': 54.0, 'heatindex_c': 13.8, 'heatindex_f': 56.8, 'dewpoint_c': 10.8, 'dewpoint_f': 51.4, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 'gust_mph': 11.8, 'gust_kph': 19.0, 'will_it_rain': 0, 'chance_of_rain': 5, 'will_it_snow': 0, 'chance_of_snow': 0}}


In [10]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1780555668,
        "localtime": "2026-06-03 23:47"
    },
    "current": {
        "last_updated_epoch": 1780554600,
        "last_updated": "2026-06-03 23:30",
        "temp_c": 13.9,
        "temp_f": 57.0,
        "is_day": 0,
        "condition": {
            "text": "Clear",
            "icon": "//cdn.weatherapi.com/weather/64x64/night/113.png",
            "code": 1000
        },
        "wind_mph": 7.8,
        "wind_kph": 12.6,
        "wind_degree": 266,
        "wind_dir": "W",
        "pressure_mb": 1016.0,
        "pressure_in": 30.01,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 89,
        "cloud": 25,
        "feelslike_c": 13.0,
        "feelslike_f": 55.3,
        "windchill_c": 12.2,
        "win

<img src="./google_sample.png" width="800" height="600">